# TokenWire Benchmark

Compare **TokenWire** (binary token streaming) vs **Baseline** (JSON text streaming) for LLM inference.

| Protocol | Format | Bytes/Token | Overhead |
|----------|--------|-------------|----------|
| **TokenWire** | Binary | 4 bytes | None |
| **Baseline** | JSON | ~60 bytes | Detokenization + Serialization |

**Just run all cells!** Everything is self-contained.

In [ ]:
#@title 1. Install Dependencies { display-mode: "form" }
#@markdown Installs llama-cpp-python with CUDA support, seaborn for charts, and other libraries.

!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121 -q
!pip install matplotlib seaborn pandas numpy -q

print("Dependencies installed!")

In [ ]:
#@title 2. TokenWire Library { display-mode: "form" }
#@markdown Core library for benchmarking. Just run this cell.

# =============================================================================
# TOKENWIRE BENCHMARK LIBRARY
# =============================================================================

import time
import json
import os
import numpy as np
import pandas as pd
from pathlib import Path
from typing import List, Dict, Any, Optional
from dataclasses import dataclass, field, asdict
from datetime import datetime
import urllib.request

# =============================================================================
# Model Registry - SLMs that work well on free Colab T4
# =============================================================================

MODEL_REGISTRY = {
    "Qwen2.5-Coder-1.5B (Recommended)": {
        "url": "https://huggingface.co/Qwen/Qwen2.5-Coder-1.5B-Instruct-GGUF/resolve/main/qwen2.5-coder-1.5b-instruct-q4_k_m.gguf",
        "filename": "qwen2.5-coder-1.5b-q4_k_m.gguf",
        "size_gb": 1.0,
        "description": "Best for coding tasks, fast inference"
    },
    "Phi-2 (2.7B)": {
        "url": "https://huggingface.co/TheBloke/phi-2-GGUF/resolve/main/phi-2.Q4_K_M.gguf",
        "filename": "phi-2-q4_k_m.gguf",
        "size_gb": 1.6,
        "description": "Microsoft's reasoning model"
    },
}

# =============================================================================
# Prompt Datasets
# =============================================================================

PROMPT_DATASETS = {
    "Mixed (Code + Reasoning + Creative)": [
        ("Write a Python function to check if a number is prime.", "Code"),
        ("Explain how a hash table works in simple terms.", "Reasoning"),
        ("Write a haiku about programming.", "Creative"),
        ("What is the time complexity of binary search?", "Reasoning"),
        ("Write a SQL query to find duplicate emails in a table.", "Code"),
        ("Explain the difference between TCP and UDP protocols.", "Reasoning"),
        ("Write a recursive factorial function in JavaScript.", "Code"),
        ("What are the SOLID principles in software design?", "Reasoning"),
        ("Write a function to reverse a singly linked list.", "Code"),
        ("Explain how garbage collection works in Java.", "Reasoning"),
        ("Write a short poem about artificial intelligence.", "Creative"),
        ("What is the difference between REST and GraphQL?", "Reasoning"),
        ("Implement binary search in Python with comments.", "Code"),
        ("Explain how HTTPS encryption protects data.", "Reasoning"),
        ("Write a function to detect a cycle in a linked list.", "Code"),
        ("What is the CAP theorem in distributed systems?", "Reasoning"),
        ("Write a merge sort implementation in Python.", "Code"),
        ("Explain microservices architecture benefits.", "Reasoning"),
        ("Write a function for longest common subsequence.", "Code"),
        ("What is eventual consistency in databases?", "Reasoning"),
        ("Create a Python decorator for timing functions.", "Code"),
        ("Explain the Observer design pattern.", "Reasoning"),
        ("Write a breadth-first search algorithm.", "Code"),
        ("What is the difference between threads and processes?", "Reasoning"),
        ("Implement a simple LRU cache in Python.", "Code"),
        ("Explain how DNS resolution works.", "Reasoning"),
        ("Write a function to validate balanced parentheses.", "Code"),
        ("What is database sharding and when to use it?", "Reasoning"),
        ("Create a simple rate limiter class.", "Code"),
        ("Explain the concept of idempotency in APIs.", "Reasoning"),
    ],
    "Code Only": [
        ("Write a Python function to check if a number is prime.", "Code"),
        ("Implement quicksort in Python.", "Code"),
        ("Write a function to find the longest palindrome substring.", "Code"),
        ("Create a binary search tree class with insert and search.", "Code"),
        ("Write a function to merge two sorted arrays.", "Code"),
        ("Implement a stack using two queues.", "Code"),
        ("Write a function to find all permutations of a string.", "Code"),
        ("Create a trie data structure for word search.", "Code"),
        ("Write a function to serialize and deserialize a binary tree.", "Code"),
        ("Implement Dijkstra's shortest path algorithm.", "Code"),
        ("Write a regex to validate email addresses.", "Code"),
        ("Create a thread-safe singleton pattern.", "Code"),
        ("Write a function to find the kth largest element.", "Code"),
        ("Implement a min heap from scratch.", "Code"),
        ("Write a function to rotate a matrix 90 degrees.", "Code"),
        ("Create a basic HTTP server in Python.", "Code"),
        ("Write a function to find all subsets of a set.", "Code"),
        ("Implement the producer-consumer pattern.", "Code"),
        ("Write a function to check if a graph is bipartite.", "Code"),
        ("Create a simple JSON parser.", "Code"),
        ("Write a function to find the median of two sorted arrays.", "Code"),
        ("Implement a basic blockchain structure.", "Code"),
        ("Write a function for topological sort.", "Code"),
        ("Create a connection pool implementation.", "Code"),
        ("Write a function to find strongly connected components.", "Code"),
        ("Implement the A* pathfinding algorithm.", "Code"),
        ("Write a simple expression evaluator.", "Code"),
        ("Create a basic pub/sub messaging system.", "Code"),
        ("Write a function to solve N-Queens problem.", "Code"),
        ("Implement a basic garbage collector concept.", "Code"),
    ],
    "Reasoning Only": [
        ("Explain how a hash table handles collisions.", "Reasoning"),
        ("What is the difference between BFS and DFS?", "Reasoning"),
        ("Explain the CAP theorem with examples.", "Reasoning"),
        ("What are the trade-offs of microservices?", "Reasoning"),
        ("Explain how HTTPS works step by step.", "Reasoning"),
        ("What is eventual consistency vs strong consistency?", "Reasoning"),
        ("Explain the difference between SQL and NoSQL.", "Reasoning"),
        ("What is the purpose of an index in databases?", "Reasoning"),
        ("Explain how load balancing works.", "Reasoning"),
        ("What is the difference between REST and gRPC?", "Reasoning"),
        ("Explain the concept of database normalization.", "Reasoning"),
        ("What are the ACID properties in databases?", "Reasoning"),
        ("Explain how OAuth 2.0 authentication works.", "Reasoning"),
        ("What is the difference between monolith and microservices?", "Reasoning"),
        ("Explain the concept of containerization.", "Reasoning"),
        ("What is the purpose of a message queue?", "Reasoning"),
        ("Explain how CDNs improve performance.", "Reasoning"),
        ("What is the difference between horizontal and vertical scaling?", "Reasoning"),
        ("Explain the concept of circuit breakers.", "Reasoning"),
        ("What is the purpose of API rate limiting?", "Reasoning"),
        ("Explain how caching strategies work.", "Reasoning"),
        ("What is the difference between optimistic and pessimistic locking?", "Reasoning"),
        ("Explain the concept of event sourcing.", "Reasoning"),
        ("What is CQRS and when to use it?", "Reasoning"),
        ("Explain how consensus algorithms work.", "Reasoning"),
        ("What is the purpose of service mesh?", "Reasoning"),
        ("Explain the concept of blue-green deployments.", "Reasoning"),
        ("What is the difference between synchronous and asynchronous APIs?", "Reasoning"),
        ("Explain how distributed tracing works.", "Reasoning"),
        ("What is the purpose of feature flags?", "Reasoning"),
    ],
}

# =============================================================================
# Model Wrapper
# =============================================================================

class TokenWireModel:
    """Llama model wrapper supporting both TokenWire and Baseline protocols."""

    def __init__(self, model_path: str, n_ctx: int = 2048, n_gpu_layers: int = -1, verbose: bool = False):
        from llama_cpp import Llama
        print(f"Loading model: {model_path}")
        self.llm = Llama(model_path=model_path, n_ctx=n_ctx, n_gpu_layers=n_gpu_layers, verbose=verbose)
        self.eos_id = self.llm.token_eos()
        self.model_path = model_path
        print(f"Model loaded. Vocab size: {self.llm.n_vocab()}, EOS: {self.eos_id}")

    def tokenize(self, text: str) -> List[int]:
        return self.llm.tokenize(text.encode("utf-8"))

    def detokenize(self, token_ids: List[int]) -> str:
        return self.llm.detokenize(token_ids).decode("utf-8", errors="replace")

    def reset_cache(self):
        self.llm.reset()

    def generate_with_metrics(self, prompt: str, protocol: str = "tokenwire", max_tokens: int = 128, temperature: float = 0.0) -> dict:
        """Generate and collect metrics."""
        self.reset_cache()
        prompt_tokens = self.tokenize(prompt)
        start_time = time.perf_counter()
        ttft = None
        tokens = []
        total_bytes = 0

        for token_id in self.llm.generate(prompt_tokens, temp=temperature):
            token_id = int(token_id)
            if token_id == self.eos_id or len(tokens) >= max_tokens:
                break
            now = time.perf_counter()
            if ttft is None:
                ttft = (now - start_time) * 1000

            if protocol == "baseline":
                token_text = self.detokenize([token_id])
                payload = json.dumps({"model": "llm", "response": token_text, "done": False})
                total_bytes += len(payload.encode('utf-8'))
            else:
                total_bytes += 4
            tokens.append(token_id)

        end_time = time.perf_counter()
        total_time = (end_time - start_time) * 1000

        return {
            'protocol': protocol,
            'ttft_ms': ttft or 0,
            'total_time_ms': total_time,
            'total_tokens': len(tokens),
            'total_bytes': total_bytes,
            'tokens_per_second': len(tokens) / (total_time / 1000) if total_time > 0 else 0,
            'bytes_per_token': total_bytes / len(tokens) if tokens else 0,
        }

# =============================================================================
# Benchmark Runner
# =============================================================================

@dataclass
class PromptData:
    id: str
    prompt: str
    category: str

@dataclass
class BenchmarkResult:
    prompt_id: str
    prompt: str
    category: str
    protocol: str
    ttft_ms: float
    total_time_ms: float
    total_tokens: int
    total_bytes: int
    tokens_per_second: float
    bytes_per_token: float

@dataclass
class BenchmarkRun:
    run_id: str
    protocol: str
    model_name: str
    started_at: str
    completed_at: Optional[str] = None
    results: List[BenchmarkResult] = field(default_factory=list)
    avg_ttft_ms: float = 0
    avg_tps: float = 0
    total_bytes: int = 0
    total_tokens: int = 0

class Benchmark:
    def __init__(self, model, max_tokens: int = 128, warmup_prompt: str = "Hello, how are you?"):
        self.model = model
        self.max_tokens = max_tokens
        self.warmup_prompt = warmup_prompt

    def warmup(self, protocol: str, runs: int = 2):
        print(f"  Warming up ({runs} runs)...")
        self.model.reset_cache()
        for _ in range(runs):
            _ = self.model.generate_with_metrics(self.warmup_prompt, protocol=protocol, max_tokens=32)
        print("  Ready.")

    def run(self, protocol: str, prompts: List[PromptData], warmup_runs: int = 2, cooldown_seconds: float = 0.2, verbose: bool = True) -> BenchmarkRun:
        model_name = Path(self.model.model_path).stem
        run_id = f"{model_name}_{protocol}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        run = BenchmarkRun(run_id=run_id, protocol=protocol, model_name=model_name, started_at=datetime.now().isoformat())

        if verbose:
            print(f"\n{'='*60}\n{protocol.upper()} BENCHMARK ({len(prompts)} prompts)\n{'='*60}")

        self.warmup(protocol, warmup_runs)

        for i, p in enumerate(prompts, 1):
            if verbose:
                print(f"[{i}/{len(prompts)}] {p.category}: {p.prompt[:45]}...")
            time.sleep(cooldown_seconds)
            metrics = self.model.generate_with_metrics(p.prompt, protocol=protocol, max_tokens=self.max_tokens)
            result = BenchmarkResult(
                prompt_id=p.id, prompt=p.prompt, category=p.category, protocol=protocol,
                ttft_ms=metrics['ttft_ms'], total_time_ms=metrics['total_time_ms'],
                total_tokens=metrics['total_tokens'], total_bytes=metrics['total_bytes'],
                tokens_per_second=metrics['tokens_per_second'], bytes_per_token=metrics['bytes_per_token']
            )
            run.results.append(result)
            if verbose:
                print(f"    TTFT: {result.ttft_ms:.1f}ms | Tokens: {result.total_tokens} | {result.total_bytes} bytes")

        run.completed_at = datetime.now().isoformat()
        run.total_tokens = sum(r.total_tokens for r in run.results)
        run.total_bytes = sum(r.total_bytes for r in run.results)
        if run.results:
            run.avg_ttft_ms = np.mean([r.ttft_ms for r in run.results])
            run.avg_tps = np.mean([r.tokens_per_second for r in run.results])

        if verbose:
            print(f"\n{protocol.upper()} COMPLETE: Avg TTFT={run.avg_ttft_ms:.1f}ms, TPS={run.avg_tps:.1f}, Total={run.total_bytes:,} bytes")
        return run

    def compare(self, baseline: BenchmarkRun, tokenwire: BenchmarkRun) -> Dict[str, Any]:
        prompt_comparisons = []
        tokenwire_wins = 0
        for b, t in zip(baseline.results, tokenwire.results):
            ttft_diff = b.ttft_ms - t.ttft_ms
            winner = "tokenwire" if ttft_diff > 0 else "baseline"
            if winner == "tokenwire":
                tokenwire_wins += 1
            prompt_comparisons.append({
                'prompt_id': b.prompt_id, 'category': b.category, 'prompt': b.prompt[:50],
                'baseline_ttft_ms': round(b.ttft_ms, 2), 'tokenwire_ttft_ms': round(t.ttft_ms, 2),
                'ttft_diff_ms': round(ttft_diff, 2), 'winner': winner,
                'baseline_bytes': b.total_bytes, 'tokenwire_bytes': t.total_bytes,
                'bandwidth_savings_pct': round((1 - t.total_bytes / b.total_bytes) * 100, 1) if b.total_bytes > 0 else 0
            })

        bandwidth_savings = (1 - tokenwire.total_bytes / baseline.total_bytes) * 100 if baseline.total_bytes > 0 else 0
        return {
            'model_name': baseline.model_name,
            'num_prompts': len(baseline.results),
            'baseline': {
                'avg_ttft_ms': round(baseline.avg_ttft_ms, 2),
                'avg_tps': round(baseline.avg_tps, 1),
                'total_bytes': baseline.total_bytes,
                'total_tokens': baseline.total_tokens,
            },
            'tokenwire': {
                'avg_ttft_ms': round(tokenwire.avg_ttft_ms, 2),
                'avg_tps': round(tokenwire.avg_tps, 1),
                'total_bytes': tokenwire.total_bytes,
                'total_tokens': tokenwire.total_tokens,
            },
            'comparison': {
                'ttft_improvement_ms': round(baseline.avg_ttft_ms - tokenwire.avg_ttft_ms, 2),
                'ttft_improvement_pct': round((1 - tokenwire.avg_ttft_ms / baseline.avg_ttft_ms) * 100, 1) if baseline.avg_ttft_ms > 0 else 0,
                'bandwidth_savings_pct': round(bandwidth_savings, 1),
                'tokenwire_wins': tokenwire_wins,
                'total_prompts': len(baseline.results),
                'tokenwire_win_rate_pct': round((tokenwire_wins / len(baseline.results)) * 100, 0) if baseline.results else 0,
            },
            'per_prompt': prompt_comparisons
        }

# =============================================================================
# Charts with Seaborn
# =============================================================================

class Charts:
    COLORS = {
        'baseline': '#e74c3c',
        'tokenwire': '#2ecc71',
    }

    def __init__(self, output_dir: str = "reports"):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True)
        self._setup_style()

    def _setup_style(self):
        import matplotlib.pyplot as plt
        import seaborn as sns
        sns.set_theme(style="whitegrid")
        plt.rcParams['figure.figsize'] = (10, 6)
        plt.rcParams['font.size'] = 11
        plt.rcParams['axes.titlesize'] = 14
        plt.rcParams['axes.titleweight'] = 'bold'

    def _save(self, fig, name: str):
        path = self.output_dir / f"{name}.png"
        fig.savefig(path, dpi=150, bbox_inches='tight', facecolor='white')
        print(f"  Saved: {path}")

    def ttft_comparison(self, baseline, tokenwire, comparison, save=True):
        import matplotlib.pyplot as plt
        import seaborn as sns

        fig, ax = plt.subplots(figsize=(8, 6))
        protocols = ['Baseline\n(JSON)', 'TokenWire\n(Binary)']
        values = [baseline.avg_ttft_ms, tokenwire.avg_ttft_ms]
        colors = [self.COLORS['baseline'], self.COLORS['tokenwire']]

        bars = ax.bar(protocols, values, color=colors, width=0.5, edgecolor='white', linewidth=2)
        for bar, val in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{val:.1f}ms',
                    ha='center', va='bottom', fontweight='bold', fontsize=12)

        improvement = comparison['comparison']['ttft_improvement_pct']
        ax.set_ylabel('Time to First Token (ms)', fontsize=12)
        ax.set_title(f'TTFT Comparison ({improvement:.1f}% faster)', fontsize=14, fontweight='bold')
        ax.set_ylim(0, max(values) * 1.25)
        sns.despine()
        plt.tight_layout()

        if save:
            self._save(fig, 'ttft_comparison')
        return fig

    def bandwidth_comparison(self, baseline, tokenwire, comparison, save=True):
        import matplotlib.pyplot as plt
        import seaborn as sns

        fig, ax = plt.subplots(figsize=(8, 6))
        protocols = ['Baseline\n(JSON)', 'TokenWire\n(Binary)']
        values = [baseline.total_bytes/1024, tokenwire.total_bytes/1024]
        colors = [self.COLORS['baseline'], self.COLORS['tokenwire']]

        bars = ax.bar(protocols, values, color=colors, width=0.5, edgecolor='white', linewidth=2)
        for bar, val in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f'{val:.1f} KB',
                    ha='center', va='bottom', fontweight='bold', fontsize=12)

        savings = comparison['comparison']['bandwidth_savings_pct']
        ax.set_ylabel('Total Bandwidth (KB)', fontsize=12)
        ax.set_title(f'Bandwidth Comparison ({savings:.1f}% reduction)', fontsize=14, fontweight='bold')
        ax.set_ylim(0, max(values) * 1.25)
        sns.despine()
        plt.tight_layout()

        if save:
            self._save(fig, 'bandwidth_comparison')
        return fig

    def ttft_per_prompt(self, baseline, tokenwire, save=True):
        import matplotlib.pyplot as plt
        import seaborn as sns

        fig, ax = plt.subplots(figsize=(12, 5))
        x = range(1, len(baseline.results) + 1)

        ax.plot(x, [r.ttft_ms for r in baseline.results], 'o-',
                color=self.COLORS['baseline'], label='Baseline', markersize=8, linewidth=2)
        ax.plot(x, [r.ttft_ms for r in tokenwire.results], 's-',
                color=self.COLORS['tokenwire'], label='TokenWire', markersize=8, linewidth=2)

        ax.fill_between(x, [r.ttft_ms for r in baseline.results], alpha=0.15, color=self.COLORS['baseline'])
        ax.fill_between(x, [r.ttft_ms for r in tokenwire.results], alpha=0.15, color=self.COLORS['tokenwire'])

        ax.set_xlabel('Prompt #', fontsize=12)
        ax.set_ylabel('TTFT (ms)', fontsize=12)
        ax.set_title('Time to First Token per Prompt', fontsize=14, fontweight='bold')
        ax.legend(loc='upper right')
        ax.set_xticks(x)
        sns.despine()
        plt.tight_layout()

        if save:
            self._save(fig, 'ttft_per_prompt')
        return fig

    def bytes_per_token(self, baseline, tokenwire, save=True):
        import matplotlib.pyplot as plt
        import seaborn as sns

        fig, ax = plt.subplots(figsize=(8, 6))
        protocols = ['Baseline\n(JSON)', 'TokenWire\n(Binary)']
        baseline_bpt = baseline.total_bytes / baseline.total_tokens if baseline.total_tokens else 0
        tokenwire_bpt = tokenwire.total_bytes / tokenwire.total_tokens if tokenwire.total_tokens else 0
        values = [baseline_bpt, tokenwire_bpt]
        colors = [self.COLORS['baseline'], self.COLORS['tokenwire']]

        bars = ax.bar(protocols, values, color=colors, width=0.5, edgecolor='white', linewidth=2)
        for bar, val in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{val:.1f}',
                    ha='center', va='bottom', fontweight='bold', fontsize=12)

        ax.set_ylabel('Bytes per Token', fontsize=12)
        ax.set_title('Protocol Efficiency: Bytes per Token', fontsize=14, fontweight='bold')
        ax.set_ylim(0, max(values) * 1.25)
        sns.despine()
        plt.tight_layout()

        if save:
            self._save(fig, 'bytes_per_token')
        return fig

    def summary_dashboard(self, baseline, tokenwire, comparison, save=True):
        import matplotlib.pyplot as plt
        import seaborn as sns

        fig = plt.figure(figsize=(14, 8))
        gs = fig.add_gridspec(2, 3, hspace=0.35, wspace=0.3)
        c = comparison['comparison']

        # 1. TTFT Bar
        ax1 = fig.add_subplot(gs[0, 0])
        vals = [baseline.avg_ttft_ms, tokenwire.avg_ttft_ms]
        bars = ax1.bar(['Baseline', 'TokenWire'], vals,
                       color=[self.COLORS['baseline'], self.COLORS['tokenwire']], edgecolor='white', linewidth=2)
        ax1.set_ylabel('TTFT (ms)')
        ax1.set_title(f'Avg TTFT\n({c["ttft_improvement_pct"]:.1f}% faster)')
        for bar in bars:
            ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{bar.get_height():.1f}',
                    ha='center', va='bottom', fontweight='bold', fontsize=10)
        sns.despine(ax=ax1)

        # 2. Bandwidth Bar
        ax2 = fig.add_subplot(gs[0, 1])
        vals = [baseline.total_bytes/1024, tokenwire.total_bytes/1024]
        bars = ax2.bar(['Baseline', 'TokenWire'], vals,
                       color=[self.COLORS['baseline'], self.COLORS['tokenwire']], edgecolor='white', linewidth=2)
        ax2.set_ylabel('KB')
        ax2.set_title(f'Total Bandwidth\n({c["bandwidth_savings_pct"]:.1f}% saved)')
        for bar in bars:
            ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{bar.get_height():.1f}',
                    ha='center', va='bottom', fontweight='bold', fontsize=10)
        sns.despine(ax=ax2)

        # 3. Win Rate Pie
        ax3 = fig.add_subplot(gs[0, 2])
        wins = c['tokenwire_wins']
        total = c['total_prompts']
        ax3.pie([wins, total-wins], labels=[f'TokenWire\n({wins})', f'Baseline\n({total-wins})'],
                colors=[self.COLORS['tokenwire'], self.COLORS['baseline']],
                autopct='%1.0f%%', startangle=90, explode=(0.05, 0),
                textprops={'fontsize': 10})
        ax3.set_title('TTFT Winner\nper Prompt')

        # 4. Bytes per Token
        ax4 = fig.add_subplot(gs[1, 0])
        baseline_bpt = baseline.total_bytes / baseline.total_tokens if baseline.total_tokens else 0
        tokenwire_bpt = tokenwire.total_bytes / tokenwire.total_tokens if tokenwire.total_tokens else 0
        bars = ax4.bar(['Baseline', 'TokenWire'], [baseline_bpt, tokenwire_bpt],
                       color=[self.COLORS['baseline'], self.COLORS['tokenwire']], edgecolor='white', linewidth=2)
        ax4.set_ylabel('Bytes')
        ax4.set_title('Bytes per Token')
        for bar in bars:
            ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{bar.get_height():.1f}',
                    ha='center', va='bottom', fontweight='bold', fontsize=10)
        sns.despine(ax=ax4)

        # 5. Throughput
        ax5 = fig.add_subplot(gs[1, 1])
        bars = ax5.bar(['Baseline', 'TokenWire'], [baseline.avg_tps, tokenwire.avg_tps],
                       color=[self.COLORS['baseline'], self.COLORS['tokenwire']], edgecolor='white', linewidth=2)
        ax5.set_ylabel('Tokens/sec')
        ax5.set_title('Throughput')
        for bar in bars:
            ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{bar.get_height():.1f}',
                    ha='center', va='bottom', fontweight='bold', fontsize=10)
        sns.despine(ax=ax5)

        # 6. Summary Text
        ax6 = fig.add_subplot(gs[1, 2])
        ax6.axis('off')
        summary = f"""Model: {comparison['model_name']}
Prompts: {c['total_prompts']}

TTFT Improvement:
  {c['ttft_improvement_ms']:.1f}ms ({c['ttft_improvement_pct']:.1f}%)

Bandwidth Saved:
  {c['bandwidth_savings_pct']:.1f}%

Win Rate:
  {c['tokenwire_win_rate_pct']:.0f}% ({wins}/{total})"""
        ax6.text(0.1, 0.5, summary, transform=ax6.transAxes, fontsize=11, verticalalignment='center',
                fontfamily='monospace', bbox=dict(boxstyle='round', facecolor='#f8f9fa', edgecolor='#dee2e6'))

        fig.suptitle('TokenWire vs Baseline: Benchmark Summary', fontsize=16, fontweight='bold', y=1.02)

        if save:
            self._save(fig, 'summary_dashboard')
        return fig

    def generate_all(self, baseline, tokenwire, comparison):
        print(f"\nSaving charts to '{self.output_dir}/'...")
        self.ttft_comparison(baseline, tokenwire, comparison)
        self.bandwidth_comparison(baseline, tokenwire, comparison)
        self.ttft_per_prompt(baseline, tokenwire)
        self.bytes_per_token(baseline, tokenwire)
        self.summary_dashboard(baseline, tokenwire, comparison)

# =============================================================================
# Report Generator
# =============================================================================

class ReportGenerator:
    def __init__(self, output_dir: str = "reports"):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True)

    def save_json(self, comparison: dict, baseline: BenchmarkRun, tokenwire: BenchmarkRun):
        report = {
            'generated_at': datetime.now().isoformat(),
            'summary': comparison,
            'baseline_run': asdict(baseline),
            'tokenwire_run': asdict(tokenwire)
        }
        path = self.output_dir / 'benchmark_report.json'
        with open(path, 'w') as f:
            json.dump(report, f, indent=2, default=str)
        print(f"  JSON: {path}")

    def save_csv(self, comparison: dict):
        df = pd.DataFrame(comparison['per_prompt'])
        path = self.output_dir / 'per_prompt_results.csv'
        df.to_csv(path, index=False)
        print(f"  CSV:  {path}")

    def save_summary(self, comparison: dict):
        c = comparison['comparison']
        summary = f"""TOKENWIRE BENCHMARK REPORT
{'='*50}
Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
Model: {comparison['model_name']}
Prompts: {comparison['num_prompts']}

RESULTS
{'-'*50}

Time to First Token (TTFT)
  Baseline:    {comparison['baseline']['avg_ttft_ms']:.2f} ms
  TokenWire:   {comparison['tokenwire']['avg_ttft_ms']:.2f} ms
  Improvement: {c['ttft_improvement_ms']:.2f} ms ({c['ttft_improvement_pct']:.1f}%)

Bandwidth
  Baseline:  {comparison['baseline']['total_bytes']:,} bytes
  TokenWire: {comparison['tokenwire']['total_bytes']:,} bytes
  Savings:   {c['bandwidth_savings_pct']:.1f}%

Win Rate
  TokenWire won {c['tokenwire_wins']}/{c['total_prompts']} prompts ({c['tokenwire_win_rate_pct']:.0f}%)

{'='*50}
"""
        path = self.output_dir / 'benchmark_summary.txt'
        with open(path, 'w') as f:
            f.write(summary)
        print(f"  TXT:  {path}")

    def generate_all(self, comparison: dict, baseline: BenchmarkRun, tokenwire: BenchmarkRun):
        print(f"\nSaving reports to '{self.output_dir}/'...")
        self.save_json(comparison, baseline, tokenwire)
        self.save_csv(comparison)
        self.save_summary(comparison)

# =============================================================================

print("Library loaded!")

In [ ]:
#@title 3. Configuration { display-mode: "form" }
#@markdown ### Model Selection
selected_model = "Qwen2.5-Coder-1.5B (Recommended)" #@param ["Qwen2.5-Coder-1.5B (Recommended)", "Phi-2 (2.7B)"]

#@markdown ### Prompt Dataset
prompt_dataset = "Mixed (Code + Reasoning + Creative)" #@param ["Mixed (Code + Reasoning + Creative)", "Code Only", "Reasoning Only"]

#@markdown ### Benchmark Settings
max_tokens = 128 #@param {type:"slider", min:32, max:256, step:32}
num_prompts = 12 #@param {type:"slider", min:4, max:30, step:2}
cooldown_between_protocols = 5 #@param {type:"slider", min:1, max:15, step:1}

# Get model info
model_info = MODEL_REGISTRY[selected_model]

print(f"Configuration:")
print(f"  Model: {selected_model}")
print(f"    Size: ~{model_info['size_gb']} GB")
print(f"    {model_info['description']}")
print(f"  Dataset: {prompt_dataset}")
print(f"  Prompts: {num_prompts}")
print(f"  Max tokens: {max_tokens}")
print(f"  Cooldown: {cooldown_between_protocols}s")

In [ ]:
#@title 4. Download Model { display-mode: "form" }
#@markdown Downloads the selected model if not present.

model_url = model_info['url']
model_filename = model_info['filename']
MODEL_PATH = f"models/{model_filename}"

if not os.path.exists(MODEL_PATH):
    print(f"Downloading {selected_model}...")
    print(f"  Size: ~{model_info['size_gb']} GB")
    os.makedirs("models", exist_ok=True)

    def progress(block_num, block_size, total_size):
        pct = min(100, block_num * block_size * 100 // total_size)
        mb = block_num * block_size / (1024*1024)
        total_mb = total_size / (1024*1024)
        print(f"\r  {pct}% ({mb:.0f}/{total_mb:.0f} MB)", end="", flush=True)

    urllib.request.urlretrieve(model_url, MODEL_PATH, progress)
    print("\nDownload complete!")
else:
    print(f"Model exists: {MODEL_PATH}")
    print(f"  Size: {os.path.getsize(MODEL_PATH) / (1024*1024):.0f} MB")

In [ ]:
#@title 5. Load Model { display-mode: "form" }
#@markdown Loads the model into GPU memory.

model = TokenWireModel(
    MODEL_PATH,
    n_ctx=2048,
    n_gpu_layers=-1
)

print("\nModel ready!")

In [ ]:
#@title 6. Prepare Prompts { display-mode: "form" }
#@markdown Loads prompts from selected dataset.

raw_prompts = PROMPT_DATASETS[prompt_dataset][:num_prompts]
prompts = [PromptData(f"p{i+1}", text, cat) for i, (text, cat) in enumerate(raw_prompts)]

categories = {}
for p in prompts:
    categories[p.category] = categories.get(p.category, 0) + 1

print(f"Loaded {len(prompts)} prompts from '{prompt_dataset}'")
print(f"\nCategories:")
for cat, count in sorted(categories.items()):
    print(f"  {cat}: {count}")

print(f"\nSample:")
for p in prompts[:3]:
    print(f"  [{p.category}] {p.prompt[:50]}...")

In [ ]:
#@title 7. Run Benchmark { display-mode: "form" }
#@markdown Runs both protocols and collects metrics.

bench = Benchmark(model=model, max_tokens=max_tokens)

# Run Baseline
baseline_run = bench.run("baseline", prompts, warmup_runs=2, verbose=True)

# Cooldown
print(f"\nCooling down for {cooldown_between_protocols}s...")
time.sleep(cooldown_between_protocols)

# Run TokenWire
tokenwire_run = bench.run("tokenwire", prompts, warmup_runs=2, verbose=True)

# Compare
comparison = bench.compare(baseline_run, tokenwire_run)

print("\n" + "="*60)
print("BENCHMARK COMPLETE!")
print("="*60)

In [ ]:
#@title 8. Results Summary { display-mode: "form" }
#@markdown Displays benchmark results.

c = comparison['comparison']

print("\n" + "="*60)
print("                    BENCHMARK RESULTS")
print("="*60)
print(f"\n  Model: {comparison['model_name']}")
print(f"  Prompts: {c['total_prompts']} | Max tokens: {max_tokens}")

print("\n" + "-"*60)
print("  TIME TO FIRST TOKEN (TTFT)")
print("-"*60)
print(f"    Baseline:    {comparison['baseline']['avg_ttft_ms']:>8.2f} ms")
print(f"    TokenWire:   {comparison['tokenwire']['avg_ttft_ms']:>8.2f} ms")
print(f"    Improvement: {c['ttft_improvement_ms']:>8.2f} ms ({c['ttft_improvement_pct']:.1f}% faster)")

print("\n" + "-"*60)
print("  BANDWIDTH")
print("-"*60)
print(f"    Baseline:  {comparison['baseline']['total_bytes']:>10,} bytes ({comparison['baseline']['total_bytes']/1024:.1f} KB)")
print(f"    TokenWire: {comparison['tokenwire']['total_bytes']:>10,} bytes ({comparison['tokenwire']['total_bytes']/1024:.1f} KB)")
print(f"    Savings:   {c['bandwidth_savings_pct']:.1f}%")

print("\n" + "-"*60)
print("  WIN RATE")
print("-"*60)
print(f"    TokenWire won {c['tokenwire_wins']}/{c['total_prompts']} prompts ({c['tokenwire_win_rate_pct']:.0f}%)")

print("\n" + "="*60)

In [ ]:
#@title 9. Generate Charts { display-mode: "form" }
#@markdown Creates visualizations with seaborn styling.

import matplotlib.pyplot as plt

charts = Charts(output_dir="reports")

# Summary Dashboard
fig = charts.summary_dashboard(baseline_run, tokenwire_run, comparison)
plt.show()

# TTFT per prompt
fig = charts.ttft_per_prompt(baseline_run, tokenwire_run)
plt.show()

# Bytes per token
fig = charts.bytes_per_token(baseline_run, tokenwire_run)
plt.show()

In [ ]:
#@title 10. Save Reports { display-mode: "form" }
#@markdown Saves all reports and charts for download.

# Save reports
reporter = ReportGenerator(output_dir="reports")
reporter.generate_all(comparison, baseline_run, tokenwire_run)

# Save remaining charts
charts.ttft_comparison(baseline_run, tokenwire_run, comparison)
charts.bandwidth_comparison(baseline_run, tokenwire_run, comparison)

# List files
print("\n" + "="*50)
print("FILES READY FOR DOWNLOAD")
print("="*50)
for f in sorted(Path("reports").glob("*")):
    size = f.stat().st_size
    if size > 1024:
        print(f"  {f.name:<30} {size/1024:.1f} KB")
    else:
        print(f"  {f.name:<30} {size} bytes")

print("\nDownload: Files panel (left) > reports/")

In [ ]:
#@title 11. Download ZIP (Optional) { display-mode: "form" }
#@markdown Creates a ZIP of all reports for easy download.

import shutil
from google.colab import files

zip_name = f"tokenwire_benchmark_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
shutil.make_archive(zip_name, 'zip', 'reports')

print(f"Created: {zip_name}.zip")
print("Downloading...")
files.download(f"{zip_name}.zip")

---

## Summary

**TokenWire achieves:**
- Lower TTFT by eliminating detokenization and JSON serialization overhead
- Significant bandwidth reduction (4 bytes vs ~60 bytes per token)

**Trade-off:** Client needs a pre-loaded vocabulary dictionary (~5-10MB per model)

---
*TokenWire - Binary token streaming for faster LLM inference*